In [1]:
# imports

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import csv
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR


In [2]:
LITE_MODE = True

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


In [4]:
# Write the test set to a CSV

with open('human_in.csv', 'w', encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for t in test[:100]:
        writer.writerow([t.summary, 0])

In [5]:
# Read it back in

human_predictions = []
with open('human_out.csv', 'r', encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        human_predictions.append(float(row[1]))

In [6]:
def human_pricer(item):
    idx = test.index(item)
    return human_predictions[idx]

In [7]:
human = human_pricer(test[0])
actual = test[0].price
print(f"Human predicted {human} for an item that actually costs {actual}")


Human predicted 120.0 for an item that actually costs 219.0


In [8]:
evaluate(human_pricer, test, size=100)

  0%|          | 0/100 [00:00<?, ?it/s]

$99 $184 $12 $15 $18 $10 $119 $135 $6 $270 $643 $329 $15 $26 $24 $18 $29 $25 $25 $53 $35 $126 $25 $127 $273 $398 $55 $6 $101 $51 $30 $5 $35 $9 $10 $419 $25 $11 $186 $33 $161 $51 $23 $155 $150 $4 $31 $18 $115 $82 $25 $111 $410 $75 $67 $34 $8 $10 $122 $28 $116 $17 $19 $60 $599 $60 $160 $355 $75 $34 $17 $2 $70 $76 $41 $9 $226 $5 $5 $4 $0 $7 $5 $74 $7 $10 $68 $74 $5 $3 $17 $45 $5 $16 $0 $153 $2 $122 $150 $355 

In [9]:
# Prepare our documents and prices

y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [10]:
# Use the HashingVectorizer for a Bag of Words model
# Using binary=True with the CountVectorizer makes "one-hot vectors"

np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

In [11]:
# Define the neural network - here is Pytorch code to create a 8 layer neural network

class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

In [12]:
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

# Create the loader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize the model
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [13]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

Number of trainable parameters: 669,249


In [14]:
# Define loss function and optimizer

loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# We will do 2 complete runs through the data

EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # The next 4 lines are the 4 stages of training: forward pass, loss calculation, backward pass, optimize
        
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [1/2], Train Loss: 13891.323, Val Loss: 19857.305


  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [2/2], Train Loss: 18457.865, Val Loss: 18081.900


In [15]:
def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

In [16]:
evaluate(neural_network, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$83 $43 $14 $12 $40 $172 $0 $45 $32 $8 $488 $159 $87 $151 $10 $13 $2 $23 $84 $20 $32 $41 $57 $2 $281 $277 $244 $27 $35 $39 $43 $177 $34 $23 $73 $323 $30 $76 $125 $54 $164 $73 $11 $125 $112 $46 $68 $40 $19 $33 $17 $55 $136 $34 $96 $77 $45 $117 $3 $36 $82 $5 $39 $8 $448 $127 $45 $302 $16 $229 $8 $29 $143 $94 $13 $46 $66 $35 $27 $63 $68 $128 $21 $44 $12 $43 $61 $150 $103 $112 $27 $70 $18 $18 $33 $83 $50 $12 $133 $228 $6 $63 $4 $48 $40 $44 $83 $260 $10 $92 $29 $96 $135 $7 $48 $133 $120 $49 $51 $30 $25 $181 $23 $0 $70 $15 $23 $169 $47 $47 $36 $116 $101 $23 $57 $25 $108 $72 $37 $36 $35 $149 $18 $133 $187 $66 $41 $326 $89 $11 $18 $189 $6 $68 $31 $144 $141 $7 $18 $15 $119 $8 $7 $28 $513 $16 $88 $24 $13 $44 $15 $22 $257 $45 $2 $16 $17 $8 $35 $92 $389 $16 $130 $33 $51 $86 $41 $56 $17 $15 $57 $64 $60 $18 $21 $18 $67 $52 $0 $7 

In [ ]:
openrouter/owl-alpha

In [17]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [18]:
print(test[0].summary)

Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.


In [19]:
messages_for(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

In [20]:
# The function for gpt-4.1-nano
import time
def owl_alpha(item):
    response = completion(model="openrouter/owl-alpha", messages=messages_for(item))
    # time.sleep(1)
    return response.choices[0].message.content

In [21]:
owl_alpha(test[0])


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



'Based on current market data, the estimated price for the **Old Blood Noise Excess V2 Distortion/Modulation Pedal** is **$229 USD**.'

In [22]:
evaluate(owl_alpha, test)


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



  0%|          | 0/200 [00:00<?, ?it/s]


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


$10 Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


$17 $15 Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


$30 $60 Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


$90 $79 Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


$125 Provider List: https://docs.litellm.ai/docs/providers


Provider List:

In [36]:
# The function for gpt-4.1-nano
import time
def llama_32(item):
    response = completion(model="ollama/llama3.2", api_base="http://localhost:11434", messages=messages_for(item))
    # time.sleep(1)
    return response.choices[0].message.content

In [37]:
evaluate(llama_32, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$50 $13 $30 $9 $20 $174 $94 $60 $12 $465 $468 $30 $329 $30 $21 $8 $9 $0 $60 $39 $14 $14 $164 $54 $267 $343 $104 $5 $41 $50 $5 $39 $139 $55 $25 $569 $20 $41 $15 $18 $175 $40 $20 $56 $90 $35 $3 $18 $66 $152 $16 $101 $226 $11 $746 $109 $8 $69 $73 $13 $107 $52 $8 $40 $379 $49 $150 $316 $25 $153 $20 $32 $30 $26 $25 $14 $226 $0 $3 $6 $55 $13 $10 $69 $13 $0 $52 $56 $50 $16 $3 $25 $5 $15 $2 $84 $16 $92 $150 $80 $50 $102 $12 $10 $204 $121 $10 $360 $23 $54 $9 $481 $219 $18 $34 $480 $25 $15 $31 $747 $9 $361 $49 $142 $131 $5 $5 $30 $39 $99 $68 $7 $25 $21 $15 $10 $75 $80 $99 $92 $5 $145 $37 $6 $74 $7 $20 $160 $136 $8 $4 $33 $22 $155 $14 $154 $71 $41 $2175 $0 $10 $18 $27 $7 $260 $17 $49 $35 $25 $5 $20 $18 $179 $7 $18 $150 $22 $32 $26 $63 $355 $35 $1900 $129 $200 $3 $53 $17 $50 $2 $15 $74 $64 $860 $25 $170 $29 $31 $21 $6 

In [38]:
# The function for gpt-4.1-nano
import time
def smollm(item):
    response = completion(model="ollama/smollm:360m", api_base="http://localhost:11434", messages=messages_for(item))
    # time.sleep(1)
    return response.choices[0].message.content

In [39]:
evaluate(smollm, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$219 $1899 $44 $60 $30 $230 $507 $124 $35 $329 $513 $343 $9 $196 $78 $32 $69 $68 $210 $54 $133 $123 $65 $174 $2695 $452 $1005 $10 $99 $60 $16 $40 $160 $80 $115 $818 $60 $55 $213 $37 $199 $79 $15 $235 $220 $24 $134 $14 $145 $201 $34 $130 $475 $105 $53 $133 $13 $30 $201 $2 $236 $122 $90 $110 $619 $110 $239 $285 $105 $75 $22 $3 $279 $2001 $49 $35 $124 $5 $17 $1 $880 $11 $30 $1 $14 $40 $18 $305 $119 $45 $8 $75 $8 $34 $8 $227 $33 $157 $269 $15903 $96 $117 $17 $139 $219 $118 $25 $310 $30 $299 $46 $21 $28 $127 $196 $30 $24 $17 $52 $102 $44 $311 $930 $165 $199 $59 $19 $1 $1 $139 $516 $136 $64 $29 $64 $14 $7897 $69 $2890 $137 $63 $249 $40 $80 $193 $2068 $155 $510 $264 $32 $9 $106 $46 $139 $81 $179 $951 $56 $30 $46 $225 $25 $35 $13 $1660 $177 $448 $310 $39 $10 $24 $48 $404 $107 $101 $129 $27 $92 $94 $7 $846 $5 $299 $248 $49 $19 $118 $66 $59 $23 $34 $47 $35 $124 $150 $1420 $169 $279 $48 $25 

In [40]:
# The function for gpt-4.1-nano
import time
def qwen3(item):
    response = completion(model="ollama/qwen3:0.6b", api_base="http://localhost:11434", messages=messages_for(item))
    # time.sleep(1)
    return response.choices[0].message.content

In [ ]:
evaluate(qwen3, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$19 $84 $47 $130 $320 $180 $129 $165 $24 $170 $643 $179 $80 $11 $71 $217 $11 $15 $130 $881 $14 $4 $30 $110 $272 $368 $145 $10 $54 $15 $5 $35 $95 $0 $435 $269 $20 $31 $34 $37 $150 $45 $25 $245 $165 $19 $132 $4 $100 $172 $1285 $70 $390 $130 $2 $116 $73 $15 $52 $28 $186 $2 $284 $170 $544 $1060 $110 $45 $525 $9 $23 $38 $180 $181 $41 $84 $276 $40 $43 $11 $45 $2 $5 $69 $2 $240 $62 $186 $180 $14 $58 $70 $100 $10 $6 $128 $4 $107 $20 $625 $40 $101 $17 $114 

In [ ]:
# The function for gpt-4.1-nano
import time
def deepseek(item):
    response = completion(model="ollama/deepseek-r1:8b", api_base="http://localhost:11434", messages=messages_for(item))
    # time.sleep(1)
    return response.choices[0].message.content

In [ ]:
evaluate(deepseek, test)